# 2 · Piyasa ve model: yapı, hedonik, model karşılaştırma

Teknik raporun 5–7. bölümleri ([technical.tr.md](../reports/technical.tr.md)).

**Ne koşar, ne kayıttan gelir:**
- **Hafif betikler bu defterde koşar.** `script = run("<betik>")` betiğin [1]–[5] hücrelerini çalıştırır: veriyi ve
  kayıtlı model çıktılarını (OOF tahminleri, SHAP dosyası) okur, hesabı yapar; kaydet hücresi [6] çalışmaz.
  Ardından `check` sonucun `metrics/` kaydıyla aynı olup olmadığını gösterir. `script.res`, yüklenen tablolar
  (`script.listings` …) ve betiğin bütün fonksiyonları sonraki hücrelerde kullanılabilir.
- **Ağır betikler kayıttan okunur.** Model eğiten ya da çapraz doğrulama koşan betikler (`HEAVY`) için
  `recorded("<betik>", docs)` `metrics/` kaydını getirir.
- **Soru, veri, yöntem ve sızıntı notları** kartlardan (`analysis/cards/`), **figürler** derleyicilerin çizdiği
  `reports/figures/`'tan gelir.

Analiz kodu betiklerin kendisidir; ortak kod `analysis/lib` ve `builders/report_lib`'den gelir, burada kopyası yok. Bir
sonucu kalıcı olarak değiştirmek için betik değişir ve zincir koşulur; defter yeni kaydı okur. Defter dosya yazmaz.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "analysis" / "lib").is_dir())
sys.path[:0] = [str(ROOT / "analysis"), str(ROOT / "builders")]

import pandas as pd
from IPython.display import Markdown

from lib.labels import lb
from lib.records import card, check, flat, recorded, run, runs, show
from report_lib import metrics_view as MV

pd.set_option("display.max_rows", 30)
pd.set_option("display.max_colwidth", 120)
SCRIPTS = ["05_segmentation", "06_hedonic", "07_lofo", "07_model_comparison", "07_final_model", "07_text_flag"]
docs = MV.load_metrics(ROOT / "metrics")
MV.check_consistency(docs)
runs(docs, SCRIPTS)

,yazıldı,veri sonu,DB,run_id
betik,,,,
05_segmentation,2026-11-22T18:55:24+03:00,2026-06-27,444123b1f1a9,NaN
06_hedonic,2026-11-22T18:55:28+03:00,2026-06-27,444123b1f1a9,NaN
07_lofo,2026-09-27T21:56:33+03:00,2026-06-27,444123b1f1a9,NaN
07_model_comparison,2026-09-27T21:49:42+03:00,2026-06-27,444123b1f1a9,601741517e09
07_final_model,2026-09-29T00:03:52+03:00,2026-06-27,444123b1f1a9,601741517e09
07_text_flag,2026-11-22T20:01:10+03:00,2026-06-27,444123b1f1a9,601741517e09


## 5. Piyasa yapısı — belirgin küme yok

### 05_segmentation
[betik](../analysis/05_segmentation.py) · [kart](../analysis/cards/05_segmentation.json)

In [2]:
Markdown(card("05_segmentation"))

**Soru:** Piyasanın yapısı ne: segment, kasa, yaş ve km'ye göre fiyat, iki markanın farkı; ilanlar belirgin kümeler oluşturuyor mu?

**Veri:** load_clean() (29.988 ilan).

**Yöntem:** Gruplara göre medyan / ortalama fiyat tabloları; BMW–Audi için Mann–Whitney U ve Cliff's δ; standartlaştırılmış sayısal özniteliklerde k=2..8 için KMeans silhouette'i.

**Bölme:** `none`

**Sızıntı:**
- aynı ilan iki tarafta: `n/a`
- hedef öznitelikte: `n/a`
- bütün veriye uydurma: `n/a`
- zamansal: `n/a`

In [3]:
script = run("05_segmentation")           # [1]–[5]: yükle + hesapla; kaydetmez
m = script.to_metrics(script.res)
check(m, docs["05_segmentation"])

silhouette by k | k'ye göre silhouette: [[2, 0.242], [3, 0.188], [4, 0.193], [5, 0.194], [6, 0.239], [7, 0.231], [8, 0.224]]


**Kayıtla aynı:** defterde hesaplanan sonuç `metrics/` kaydıyla birebir aynı.

In [4]:
d = m["domain"]
show("segment merdiveni", pd.DataFrame(d["segment_ladder"], columns=["segment", "medyan fiyat", "ilan"]))
show("kasa tipine göre", pd.DataFrame(d["body_median"], columns=["kasa tipi", "medyan fiyat", "ilan"]))
show("yaşa göre fiyat", pd.DataFrame(d["age_depreciation"], columns=["yaş", "medyan fiyat", "ilan", "ortalama fiyat"]))
show("km'ye göre fiyat", pd.DataFrame(d["km_price"], columns=["km kovası", "medyan fiyat", "ilan", "ortalama fiyat"]))
show("km kapsamı, marka karşılaştırması, not",
     flat({"km_price_scope": d["km_price_scope"], "brand_compare": d["brand_compare"], "age_km_note": d["age_km_note"]}))
show("KMeans: k seçimi", pd.DataFrame(m["methodology"]["kmeans_selection"]["silhouette"],
                                      columns=["k", "silhouette"]))

**segment merdiveni**

,segment,medyan fiyat,ilan
0,B,939500.0,127
1,C,1310000.0,9190
2,D,1525000.0,12357
3,E,1939875.0,7870
4,S,2490000.0,119
5,F,2675000.0,325


**kasa tipine göre**

,kasa tipi,medyan fiyat,ilan
0,Hatchback/3,830000.0,844
1,Hatchback/5,1280000.0,5829
2,MPV,1415000.0,227
3,Sedan,1600000.0,20355
4,Station wagon,1645000.0,485
5,Coupe,2034750.0,1856
6,missing,2575000.0,291


**yaşa göre fiyat**

,yaş,medyan fiyat,ilan,ortalama fiyat
0,1.0,4230000.0,600.0,4390141.0
1,2.0,4100000.0,781.0,4147958.0
2,3.0,3725000.0,1266.0,3672367.0
3,4.0,3375000.0,1154.0,3363206.0
4,5.0,2650000.0,1137.0,2799083.0
5,6.0,2450000.0,1455.0,2554266.0
6,7.0,2470000.0,674.0,2445009.0
7,8.0,2125000.0,1165.0,2198889.0
8,9.0,1927500.0,2074.0,2069137.0
9,10.0,1650000.0,3072.0,1757007.0


... (2 outputs left out)

**km kapsamı, marka karşılaştırması, not**

,değer
km_price_scope.upper_limit,400000
km_price_scope.outside,653
brand_compare.bmw_median,1575000.0
brand_compare.audi_median,1475000.0
brand_compare.bmw_n,17896
brand_compare.audi_n,12092
brand_compare.bmw_mean,1910680.084879
brand_compare.audi_mean,1822023.297469
brand_compare.mwu_p,0.0
brand_compare.cliffs_delta,0.0663


**KMeans: k seçimi**

,k,silhouette
0,2,0.242
1,3,0.188
2,4,0.193
3,5,0.194
4,6,0.239
5,7,0.231
6,8,0.224


![tr-01-body-median](../reports/figures/tr-01-body-median.png)

![tr-05-age-price](../reports/figures/tr-05-age-price.png)

![tr-06-km-price](../reports/figures/tr-06-km-price.png)

![tr-07-brand](../reports/figures/tr-07-brand.png)

## 6. Hedonik model — kontrollü etkiler

### 06_hedonic
[betik](../analysis/06_hedonic.py) · [kart](../analysis/cards/06_hedonic.json)

In [5]:
Markdown(card("06_hedonic"))

**Soru:** Her sürücünün (yaş, km, motor, hasar) fiyata kontrollü etkisi ne kadar, model kimliği sabitlenince nasıl değişiyor ve bu tahmin ne kadar güvenilir?

**Veri:** load_clean() (29.988 ilan); motor gücü ya da hacmi eksik 434 ilan OLS'tan çıkar (29.554).

**Yöntem:** Log fiyat üzerine iki OLS (yaş ve km medyan araca ortalanmış, kare ve etkileşim terimleri): segment kontrolü (segment, marka, yakıt, vites kuklaları) ve model kontrolü (aynısı, model adı model içi dönüşümle emilmiş: sabit etkiler kestiricisi); %95 güven aralıkları modele göre kümelenmiş standart hatalardan; iki sütun arasındaki farkın terim başına kümeli testi; duyarlılıklar (seriye göre kümeleme, §8'in motor değeri tutarsız ilanları olmadan model sütunu, gücün model içi değişimi); segment sütununun en yüksek VIF'i, Breusch–Pagan, genel hp–cc Pearson korelasyonu.

**Bölme:** `none`

**Sızıntı:**
- aynı ilan iki tarafta: `n/a`
- hedef öznitelikte: `none` — Fiyat yalnız bağımlı değişken; açıklayıcılarda fiyattan türeyen terim yok.
- bütün veriye uydurma: `present` — Tasarım gereği: açıklama modeli bütün veride kuruluyor, tahmin performansı iddiası yok; çıkarım modele göre kümelenmiş standart hatalarla.
- zamansal: `n/a`

In [6]:
script = run("06_hedonic")           # [1]–[5]: yükle + hesapla; kaydetmez
m = script.to_metrics(script.res)
check(m, docs["06_hedonic"])

segment R² 0.9312 [('age', np.float64(-6.64), np.float64(-6.92), np.float64(-6.37)), ('age_sq', np.float64(0.08), np.float64(0.04), np.float64(0.13)), ('km100k', np.float64(-15.11), np.float64(-15.81), np.float64(-14.4)), ('km_sq', np.float64(1.56), np.float64(1.08), np.float64(2.05)), ('age_x_km', np.float64(-0.62), np.float64(-0.89), np.float64(-0.35)), ('heavy_damage', np.float64(-11.6), np.float64(-12.51), np.float64(-10.68)), ('painted', np.float64(-1.05), np.float64(-1.17), np.float64(-0.92)), ('changed', np.float64(-3.06), np.float64(-3.33), np.float64(-2.79)), ('hp100', np.float64(19.86), np.float64(13.89), np.float64(26.15)), ('litre', np.float64(7.49), np.float64(0.71), np.float64(14.73))]
model R² 0.9648 [('age', np.float64(-5.8), np.float64(-6.33), np.float64(-5.27)), ('age_sq', np.float64(0.03), np.float64(-0.03), np.float64(0.08)), ('km100k', np.float64(-14.83), np.float64(-15.31), np.float64(-14.34)), ('km_sq', np.float64(1.67), np.float64(1.34), np.float64(2.01)), ('age

**Kayıtla aynı:** defterde hesaplanan sonuç `metrics/` kaydıyla birebir aynı.

In [7]:
hr = m["domain"]["hedonic_reliability"]
show("manşet (model kontrolü)", flat(m["domain"]["hedonic"]))
show("güvenilirlik: özet", flat(hr))
for c, col in hr["columns"].items():
    show(f"katsayılar: {c} kontrolü", pd.DataFrame(col["coefficients"]))
show("iki sütun arasındaki fark", pd.DataFrame(hr["difference"]))
show("motor aykırıları çıkınca", pd.DataFrame(hr["without_spec_outliers"]["coefficients"]))
show("örneklemden düşenler", flat(m["error_drivers"]["hedonic_dropped"]))

**manşet (model kontrolü)**

,değer
r2,0.9648
control,model
age_pct,-5.8
damage_pct,-12.46
km100k_pct,-14.83
hp100_pct,19.3
cc_litre_pct,5.97
note,Model kontrolü sütunu: etkiler aynı model adı içinde. Dönem etkisi modelde yok (kullanıcı kararı): dönemler havuzlan...


**güvenilirlik: özet**

,değer
n,29554
cluster,model
n_clusters,735
center.age,11.0
center.km,181000.0
columns.segment.r2,0.9312
columns.model.r2,0.9648
series_cluster.n_clusters,22
series_cluster.zero_in_ci.segment,litre
series_cluster.zero_in_ci.model,"age_sq, age_x_km, litre"


**katsayılar: segment kontrolü**

,term,point,se,ci_lo,ci_hi,pct_effect,pct_lo,pct_hi,contains_zero
0,age,-0.0687,0.0015,-0.0717,-0.0658,-6.64,-6.92,-6.37,False
1,age_sq,0.0008,0.0002,0.0004,0.0013,0.08,0.04,0.13,False
2,km100k,-0.1638,0.0042,-0.1721,-0.1555,-15.11,-15.81,-14.40,False
3,km_sq,0.0155,0.0024,0.0107,0.0203,1.56,1.08,2.05,False
4,age_x_km,-0.0062,0.0014,-0.0090,-0.0035,-0.62,-0.89,-0.35,False
5,heavy_damage,-0.1233,0.0053,-0.1336,-0.1130,-11.60,-12.51,-10.68,False
6,painted,-0.0105,0.0006,-0.0118,-0.0092,-1.05,-1.17,-0.92,False
7,changed,-0.0311,0.0014,-0.0339,-0.0283,-3.06,-3.33,-2.79,False
8,hp100,0.1812,0.0260,0.1301,0.2323,19.86,13.89,26.15,False
9,litre,0.0723,0.0332,0.0071,0.1374,7.49,0.71,14.73,False


... (4 outputs left out)

**motor aykırıları çıkınca**

,term,point,se,ci_lo,ci_hi,pct_effect,pct_lo,pct_hi,contains_zero
0,hp100,0.2311,0.0614,0.1105,0.3518,26.00,11.68,42.16,False
1,litre,0.0635,0.0337,-0.0027,0.1296,6.55,-0.27,13.83,True


**örneklemden düşenler**

,değer
hp,426
cc,356
both,348
total,434


![tr-03-hedonic-ci](../reports/figures/tr-03-hedonic-ci.png)

### 07_lofo
[betik](../analysis/07_lofo.py) · [kart](../analysis/cards/07_lofo.json)

In [8]:
Markdown(card("07_lofo"))

**Soru:** Model hangi öznitelik ya da öznitelik grubu olmadan ne kadar kötüleşir (başkalarının telafi edemediği bilgi)?

**Veri:** load_clean() (29.988 ilan), manşet modelin öznitelikleri.

**Yöntem:** Manşet LightGBM aynı 5-fold OOF ve parametrelerle, her sayısal öznitelik ve üç öznitelik grubu (hasar, motor, model/seri metni) tek tek çıkarılarak yeniden koşulur; RMSE artışı ve fold başına ağaç sayısı yazılır.

**Bölme:** `kfold_oof`

**Sızıntı:**
- aynı ilan iki tarafta: `present` — Farklı ad_id ile yeniden ilan edilen aynı araç fold'lar arasına düşebilir: 01_dedup_leakage katı tanımda %0,46, fiyatsız tanımda %4,26 ölçüyor; etkisi ölçülmedi.
- hedef öznitelikte: `none` — Öznitelik listesinde (lib/common.FEATURES) fiyat ya da fiyattan türeyen bir alan yok.
- bütün veriye uydurma: `present` — Kategori kodları, TF-IDF/SVD ve tabanlar yalnız eğitim fold'unda kurulur (lib/cv.fold_matrices). Ama LightGBM'in erken durdurması her fold'un kendi doğrulama kısmında (lib/cv.fit_lgb_fold): ağaç sayısı skorlanan veriyle seçiliyor, OOF hatası hafifçe iyimser olabilir. Etkisi ölçülmedi (iç içe bölmeyle ölçülebilir).
- zamansal: `present` — KFold dönemleri karıştırır; model zamanı görmez. Son taramada yeni ilanlardaki hata 08_forward_test'te ayrıca ölçülüyor.

In [9]:
m = recorded("07_lofo", docs)

**Kayıttan okunuyor** (ağır betik: her öznitelik çıkarımı için 5-fold çapraz doğrulama (~15 dk)).

In [10]:
lo = m["methodology"]
show("bir öznitelik (ya da grup) çıkınca RMSE artışı",
     pd.DataFrame([[lb(n, "tr"), n, d, kind] for n, d, kind in lo["lofo"]],
                  columns=["öznitelik", "kimlik", "ΔRMSE (₺)", "tür"]))
show("fold başına ağaç sayısı", pd.DataFrame(lo["lofo_trees"], index=[f"fold {i}" for i in range(1, 6)]).T)

**bir öznitelik (ya da grup) çıkınca RMSE artışı**

,öznitelik,kimlik,ΔRMSE (₺),tür
0,Kilometre,gb_mileage,54922.4,single
1,Yaş (yıl),vehicle_age,46049.1,single
2,DAMAGE_COLS,DAMAGE_COLS,25473.2,group
3,MODEL_SERIES,MODEL_SERIES,19413.0,group
4,model,model,15398.9,single
5,Ağır hasar,is_heavy_damaged,5207.3,single
6,Kaput,hood_state,2774.4,single
7,series,series,2071.1,single
8,Çamurluk değişen,fender_changed,1851.1,single
9,Kapı boyalı,door_painted,1250.5,single


**fold başına ağaç sayısı**

,fold 1,fold 2,fold 3,fold 4,fold 5
baseline,274,258,212,212,307
vehicle_age,314,434,330,379,335
gb_mileage,127,146,107,115,143
power_hp_val,227,180,235,163,236
engine_cc_val,224,199,239,213,234
door_changed,204,247,254,245,230
door_painted,218,186,231,220,235
door_local,200,234,250,225,304
fender_changed,214,190,200,187,220
fender_painted,223,208,226,219,245


![tr-04-lofo-flat](../reports/figures/tr-04-lofo-flat.png)

## 7. Model karşılaştırma ve kısıtlar

### 07_model_comparison
[betik](../analysis/07_model_comparison.py) · [kart](../analysis/cards/07_model_comparison.json)

In [11]:
Markdown(card("07_model_comparison"))

**Soru:** Hangi model fiyatı en iyi tahmin ediyor ve basit bir tabanı ne kadar geçiyor?

**Veri:** load_clean(): TR plakalı, fiyatı > 0, ad_id başına son kayıt (29.988 ilan).

**Yöntem:** LightGBM (model/seri TF-IDF + SVD, fold içinde kurulur) ve üç CatBoost varyantı, aynı 5-fold OOF; iki medyan tabanı (seri + yıl; model + yıl) aynı fold'larda. OOF artefaktını sonraki betikler için yazar.

**Bölme:** `kfold_oof`

**Sızıntı:**
- aynı ilan iki tarafta: `present` — Farklı ad_id ile yeniden ilan edilen aynı araç fold'lar arasına düşebilir: 01_dedup_leakage katı tanımda %0,46, fiyatsız tanımda %4,26 ölçüyor; etkisi ölçülmedi.
- hedef öznitelikte: `none` — Öznitelik listesinde (lib/common.FEATURES) fiyat ya da fiyattan türeyen bir alan yok.
- bütün veriye uydurma: `present` — Kategori kodları, TF-IDF/SVD ve tabanlar yalnız eğitim fold'unda kurulur (lib/cv.fold_matrices). Ama LightGBM'in erken durdurması her fold'un kendi doğrulama kısmında (lib/cv.fit_lgb_fold): ağaç sayısı skorlanan veriyle seçiliyor, OOF hatası hafifçe iyimser olabilir. Etkisi ölçülmedi (iç içe bölmeyle ölçülebilir).
- zamansal: `present` — KFold dönemleri karıştırır; model zamanı görmez. Son taramada yeni ilanlardaki hata 08_forward_test'te ayrıca ölçülüyor.

In [12]:
m = recorded("07_model_comparison", docs)

**Kayıttan okunuyor** (ağır betik: LightGBM ve iki CatBoost ile 5-fold çapraz doğrulama; OOF kaydını yazar).

In [13]:
d = m["domain"]
fr = d["final_results"]["model_comparison"]
show("tabanlar ve model varyantları (5-fold OOF)",
     pd.DataFrame({k: fr[k] for k in ("naive", "dealer", "lightgbm_tfidf_svd", "catboost_tfidf_svd", "catboost_native")}).T)
show("kazanan, eğitim, notlar", flat({"winner": fr["winner"], "note": fr["note"], "training": d["final_results"]["training"],
                                       "model_compare_note": d["model_compare"]["note"]}))
show("emsal medyanı tabanı", flat({k: v for k, v in d["model_year_median"].items() if k not in ("coverage", "metric_breakdown")}))
show("taban merdiveni: basamak başına", pd.DataFrame(d["model_year_median"]["metric_breakdown"],
                                                       columns=["basamak", "ilan", "pay %", "MAPE", "MAE", "R²"]))
show("emsal kapsamı", flat(d["dealer_coverage"]))
show("emsali olan ilanlarda taban ve model", flat(m["error_drivers"]["baseline_equal_terms"]))
show("yeniden üretim", flat(m["meta"]["repro"]))

**tabanlar ve model varyantları (5-fold OOF)**

,MAE,MedAE,RMSE,MAPE,R2
naive,203621.0,142500.0,308976.0,11.65,0.9216
dealer,191224.0,130000.0,305050.0,11.20,0.9235
lightgbm_tfidf_svd,109776.0,75320.0,176225.0,6.49,0.9745
catboost_tfidf_svd,110085.0,74927.0,176130.0,6.44,0.9745
catboost_native,112925.0,77660.0,178312.0,6.58,0.9739


**kazanan, eğitim, notlar**

,değer
winner,catboost
note,"İKİ medyan tabanı + ÜÇ model varyantı, hepsi aynı 5-fold OOF ile. Tabanlar: naive (seri+yıl medyanı = kaba arama), d..."
training.n_listings,29988
training.n_feature,25
training.target,log1p(price)
model_compare_note,"CatBoost iki şekilde: TF-IDF-SVD beslemeli (LightGBM ile aynı girdi, adil karşılaştırma) ve kendi native text motoru..."


**emsal medyanı tabanı**

,değer
baseline.MAE,191224.0
baseline.MedAE,130000.0
baseline.RMSE,305050.0
baseline.MAPE,11.2
baseline.R2,0.9235
model.MAE,109776.0
model.MedAE,75320.0
model.RMSE,176225.0
model.MAPE,6.49
model.R2,0.9745


... (4 outputs left out)

**emsali olan ilanlarda taban ve model**

,değer
listings,29236
pct,97.49
baseline_mae,178897.0
model_mae,105984.0
gap_tl,72913.0
improvement_pct,40.8
note,Emsali olan ilanlar = taban merdiveninin model+yıl basamağı (fold içi).


**yeniden üretim**

,değer
seed,42
catboost_device,CPU
lgb_deterministic,True
row_order,ORDER BY ad_id
n_jobs,16
cv_trees,"274, 258, 212, 212, 307"


![tr-00-base-vs-model](../reports/figures/tr-00-base-vs-model.png)

### 07_final_model
[betik](../analysis/07_final_model.py) · [kart](../analysis/cards/07_final_model.json)

In [14]:
Markdown(card("07_final_model"))

**Soru:** Serviste kullanılan son modeller hangileri ve nasıl eğitiliyor?

**Veri:** load_clean() (29.988 ilan) + 07_model_comparison'ın OOF artefaktı (ağaç sayıları, OOF artıkları).

**Yöntem:** LightGBM (ağaç = CV medyanı), CatBoost ve native CatBoost bütün ilanlarda eğitilir; servis dosyaları yazılır.

**Bölme:** `none`

**Sızıntı:**
- aynı ilan iki tarafta: `n/a`
- hedef öznitelikte: `none` — Öznitelik listesinde (lib/common.FEATURES) fiyat ya da fiyattan türeyen bir alan yok.
- bütün veriye uydurma: `present` — Tasarım gereği: servis edilen model bütün veride eğitilir. Raporun bütün doğruluk sayıları OOF'tan gelir; bu betik rapora yalnız ağaç kuralını verir.
- zamansal: `n/a`

In [15]:
m = recorded("07_final_model", docs)

**Kayıttan okunuyor** (ağır betik: servis modellerini bütün veriyle eğitir ve yazar).

In [16]:
show("final model", flat(m["meta"]["repro"]))

**final model**

,değer
final_lgb_trees,258


### 07_text_flag
[betik](../analysis/07_text_flag.py) · [kart](../analysis/cards/07_text_flag.json)

In [17]:
Markdown(card("07_text_flag"))

**Soru:** İlan metninde dönüşüm ya da modifiye geçen ilanlarda model daha çok mu yanılıyor, başka etkenler sabitken de?

**Veri:** load_clean() (29.988 ilan), ilan metinleri ve 07_model_comparison'ın OOF tahminleri (data/analysis/oof.parquet).

**Yöntem:** Metin bayrağı (dönüşüm/modifiye ifadesi) kurulur; bayraklı ve öteki ilanların büyük hata oranı ve medyan hatası karşılaştırılır; büyük hatanın bayrağa yaş bandı, log km, fiyat çeyreği, performans ailesi, log emsal ve marka kontrollü lojistik regresyonu.

**Bölme:** `oof_input`

**Sızıntı:**
- aynı ilan iki tarafta: `n/a`
- hedef öznitelikte: `none` — Bayrak ilan metninden, fiyattan değil; hata OOF tahminden.
- bütün veriye uydurma: `present` — Tasarım gereği: lojistik regresyon bütün veride bir ilişki testi, tahmin iddiası yok.
- zamansal: `n/a`

In [18]:
script = run("07_text_flag")           # [1]–[5]: yükle + hesapla; kaydetmez
m = script.to_metrics(script.res)
check(m, docs["07_text_flag"])

flagged | bayraklı: 3107 · OR 1.14


**Kayıtla aynı:** defterde hesaplanan sonuç `metrics/` kaydıyla birebir aynı.

In [19]:
show("metin bayrağı ve büyük hata", flat(m["error_drivers"]["text_flag"]))

**metin bayrağı ve büyük hata**

,değer
controlled.or,1.14
controlled.ci_lo,0.94
controlled.ci_hi,1.37
controlled.p,0.174
controlled.perf_share_pct,43.6
controlled.controls,"yas bandi, log km, fiyat ceyregi, performans ailesi, log emsal, marka"
n,3107
pct,10.36
big_pct,4.9
other_big_pct,3.9
